# 4. Recalibration & Smoothed Calibration Plots

## Prerequisites
**Run `2_load_CKD_model.ipynb` first** in this kernel session before running any cell here.

The following objects must already be in memory:

| Variable | Description |
|---|---|
| `pred_test_A`, `pred_test_B` | `dict[model_name → np.ndarray]` — internal test predictions |
| `pred_ext_A`, `pred_ext_B` | `dict[model_name → np.ndarray]` — external validation predictions |
| `y_test_A`, `y_test` | internal test labels (Scenario A / B) |
| `y_ext_A`, `y_ext` | external validation labels (Scenario A / B) |
| `OUTDIR`, `MODEL_ORDER` | output directory and model name list |

---

## What this notebook does

### 1. Smoothed calibration plots (replaces binned `calibration_curve`)
- LOWESS smoothing (frac=0.30) with **pointwise 95% bootstrap CI** (n=500)
- X-axis zoomed to 0–0.30 to properly show the 0–15% prediction range
- Inline histogram of predicted probabilities (rug)

### 2. Intercept recalibration
- **Why needed**: external cohort has lower baseline incidence (2.0% vs 4.2%).  
  The model was calibrated to internal prevalence and over-predicts externally.
- **Method**: intercept-only logistic recalibration (`logit(p_recal) = a + 1·logit(p_raw)`).  
  Intercept `a` is fit on **internal test** predictions+outcomes via GLM with offset.
  This estimates the calibration intercept within the development setting,
  then the same `a` is subtracted when applied externally to shift the probability scale.
- **Separate recalibration per scenario** (A, B) since the underlying prediction distributions differ.

> **Methodological note**: Ideally recalibration should be done on a held-out *calibration set*
> (not the same set used to evaluate discrimination).  Here we use the internal test set as a
> pragmatic approximation — this is standard practice in single-centre cohort studies and should
> be noted as a limitation.  A more rigorous approach would use cross-validated predictions on
> the full development cohort.

### 3. Calibration metrics table
- Calibration intercept (`E/O` ratio equivalent)
- Calibration slope (logistic recalibration with free slope)
- ICI (Integrated Calibration Index)
- E50/E90/Emax (calibration error quantiles)
- Brier score

In [ ]:
%run /home/ubuntu/xgboost_CKD/2_load_CKD_model.ipynb

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm
import warnings
from pathlib import Path

OUTDIR = Path('/home/ubuntu/xgboost_CKD_output')

from scipy.special import logit, expit
from scipy.stats import norm
from sklearn.metrics import brier_score_loss
from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings("ignore", category=UserWarning)

# ── Output directory for recalibration results ────────────────────────────────
RECAL_DIR = OUTDIR / "recalibration"
RECAL_DIR.mkdir(exist_ok=True)

print("[OK] Imports complete")
print(f"[OK] Output directory: {RECAL_DIR}")

## Helper functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# A. Calibration metrics
# ═══════════════════════════════════════════════════════════════════════════════

def calibration_metrics(y_true, p_pred, frac_lowess=0.30):
    """
    Comprehensive calibration metrics for a single model.

    Returns
    -------
    dict with keys:
        intercept  – calibration intercept (logistic GLM, slope fixed=1)
        slope      – calibration slope     (logistic GLM, intercept+slope free)
        brier      – Brier score
        ici        – Integrated Calibration Index (mean |LOWESS - diagonal|)
        e50/e90/emax – calibration error quantiles
    """
    y   = np.asarray(y_true, dtype=float)
    p   = np.clip(np.asarray(p_pred, dtype=float), 1e-7, 1 - 1e-7)
    lp  = logit(p)

    # ── Calibration intercept (slope fixed = 1) ───────────────────────────────
    glm_int = sm.GLM(
        y,
        np.ones((len(y), 1)),          # design matrix = intercept only
        family=sm.families.Binomial(),
        offset=lp
    ).fit(disp=False)
    calib_intercept = float(glm_int.params[0])

    # ── Calibration slope (intercept + slope) ─────────────────────────────────
    X2 = sm.add_constant(lp)           # [intercept, logit(p)]
    glm_slope = sm.GLM(
        y, X2,
        family=sm.families.Binomial()
    ).fit(disp=False)
    calib_slope = float(glm_slope.params[1])

    # ── LOWESS-based ICI and Exx ──────────────────────────────────────────────
    sort_idx  = np.argsort(p)
    p_s, y_s  = p[sort_idx], y[sort_idx]
    smooth    = lowess(y_s, p_s, frac=frac_lowess, it=3, return_sorted=True)
    p_sm, y_sm = smooth[:, 0], smooth[:, 1]

    # Interpolate LOWESS back to original prediction grid for ICI
    y_hat = np.interp(p, p_sm, y_sm)
    abs_err = np.abs(y_hat - p)
    ici  = float(np.mean(abs_err))
    e50  = float(np.percentile(abs_err, 50))
    e90  = float(np.percentile(abs_err, 90))
    emax = float(np.max(abs_err))

    return dict(
        intercept = round(calib_intercept, 4),
        slope     = round(calib_slope, 4),
        brier     = round(brier_score_loss(y, p), 4),
        ici       = round(ici, 4),
        e50       = round(e50, 4),
        e90       = round(e90, 4),
        emax      = round(emax, 4),
    )


def calibration_metrics_table(pred_dict, y_true, label=""):
    """Return a DataFrame of calibration metrics for all models in pred_dict."""
    rows = []
    for m, p in pred_dict.items():
        row = {"Model": m}
        row.update(calibration_metrics(y_true, p))
        rows.append(row)
    df = pd.DataFrame(rows)
    if label:
        print(f"\n{'─'*60}")
        print(f"Calibration metrics — {label}")
        print(f"{'─'*60}")
        print(df.to_string(index=False))
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# B. Intercept-only recalibration
# ═══════════════════════════════════════════════════════════════════════════════

def fit_intercept_recalibration(y_train, p_train):
    """
    Fit logistic recalibration with slope fixed at 1 (intercept-only).

    Model: logit(p_recal) = a + 1·logit(p_raw)
    Fitted on (y_train, p_train) via statsmodels GLM with offset.

    Returns
    -------
    intercept : float   — fitted calibration intercept
    result    : GLMResultsWrapper
    """
    p = np.clip(np.asarray(p_train, dtype=float), 1e-7, 1 - 1e-7)
    y = np.asarray(y_train, dtype=float)
    glm = sm.GLM(
        y,
        np.ones((len(y), 1)),
        family=sm.families.Binomial(),
        offset=logit(p)
    )
    result = glm.fit(disp=False)
    return float(result.params[0]), result


def apply_intercept_recalibration(p_raw, intercept):
    """
    Apply fitted intercept to new predictions.
    p_recal = expit(intercept + logit(p_raw))
    """
    p = np.clip(np.asarray(p_raw, dtype=float), 1e-7, 1 - 1e-7)
    return expit(intercept + logit(p))


def recalibrate_pred_dict(pred_dict_train, y_train, pred_dict_target):
    """
    Fit intercept on (pred_dict_train, y_train), apply to pred_dict_target.

    Returns
    -------
    recal_dict  : dict[model → recalibrated probabilities]
    intercepts  : dict[model → fitted intercept value]
    """
    recal_dict  = {}
    intercepts  = {}
    for m in pred_dict_target:
        p_train = pred_dict_train.get(m)
        if p_train is None:
            print(f"  [SKIP] {m} — not in training pred_dict")
            continue
        intercept, _ = fit_intercept_recalibration(y_train, p_train)
        recal_dict[m] = apply_intercept_recalibration(pred_dict_target[m], intercept)
        intercepts[m] = round(intercept, 4)
        print(f"  [{m}] intercept = {intercept:+.4f}  "
              f"(mean_raw={pred_dict_target[m].mean():.4f} → "
              f"mean_recal={recal_dict[m].mean():.4f})")
    return recal_dict, intercepts


# ═══════════════════════════════════════════════════════════════════════════════
# C. Smoothed calibration plot (LOWESS + bootstrap CI)
# ═══════════════════════════════════════════════════════════════════════════════

# Visual style — mirrors the DCA / ROC colour scheme already used in Notebook 1
_STYLE_MAP = {
    "base":                dict(ls="-",  lw=1.8, color="#888780"),
    "base_lsm":            dict(ls="-",  lw=2.5, color="#1D9E75"),
    "base_lsm_cap":        dict(ls="-",  lw=2.5, color="#0F6E56"),
    "base_inflam":         dict(ls="--", lw=1.8, color="#7F77DD"),
    "base_inflam_lsm":     dict(ls="--", lw=2.5, color="#534AB7"),
    "base_inflam_lsm_cap": dict(ls="--", lw=2.5, color="#3C3489"),
}


def smooth_calib_plot(
    pred_dict,
    y_true,
    title,
    frac        = 0.30,
    n_bootstrap = 500,
    ci          = 0.95,
    xlim        = (0.0, 0.25),
    ylim        = (-0.02, 0.40),
    show_rug    = True,
    save_path   = None,
):
    """
    Smoothed calibration plot using LOWESS with bootstrap 95% CI bands.

    Parameters
    ----------
    pred_dict   : dict[model_name → np.ndarray of predicted probabilities]
    y_true      : array-like of 0/1 labels
    title       : plot title
    frac        : LOWESS bandwidth (fraction of data used per local regression)
    n_bootstrap : number of bootstrap resamples for CI
    ci          : confidence level (0.95 → 95% CI)
    xlim        : x-axis limits; default zoomed to 0–25% for low-prevalence CKD data
    ylim        : y-axis limits
    show_rug    : if True, add a rug plot (jittered) for predicted probabilities
    save_path   : if provided, save figure to this path
    """
    y_arr = np.asarray(y_true, dtype=float)

    fig, (ax_main, ax_hist) = plt.subplots(
        2, 1,
        figsize=(8, 8),
        gridspec_kw={"height_ratios": [5, 1], "hspace": 0.04},
        sharex=True,
    )

    # ── Diagonal reference ────────────────────────────────────────────────────
    x_ref = np.linspace(*xlim, 300)
    ax_main.plot(x_ref, x_ref, "k--", lw=1.2, alpha=0.55, label="Perfect calibration")

    rng    = np.random.default_rng(42)
    alpha  = (1 - ci) / 2
    x_grid = np.linspace(xlim[0] + 0.001, xlim[1], 300)

    for m, p_raw in pred_dict.items():
        p   = np.clip(np.asarray(p_raw, dtype=float), 1e-7, 1 - 1e-7)
        sty = _STYLE_MAP.get(m, dict(ls="-", lw=2, color="steelblue"))

        # Filter to x_grid range for LOWESS (avoids edge effects)
        mask     = (p >= xlim[0]) & (p <= xlim[1])
        p_m, y_m = p[mask], y_arr[mask]

        if len(p_m) < 20:
            print(f"  [WARN] {m}: only {len(p_m)} points in xlim range — skipping LOWESS")
            continue

        # Central LOWESS curve
        sort_idx   = np.argsort(p_m)
        smooth_pts = lowess(y_m[sort_idx], p_m[sort_idx],
                            frac=frac, it=3, return_sorted=True)
        # Clamp observed rates to [0, 1]
        smooth_pts[:, 1] = np.clip(smooth_pts[:, 1], 0, 1)

        # Bootstrap CI
        boot_curves = np.empty((n_bootstrap, len(x_grid)))
        for b in range(n_bootstrap):
            idx_b     = rng.integers(0, len(p_m), len(p_m))
            p_b, y_b  = p_m[idx_b], y_m[idx_b]
            sort_b    = np.argsort(p_b)
            sm_b      = lowess(y_b[sort_b], p_b[sort_b],
                               frac=frac, it=1, return_sorted=True)
            sm_b[:, 1] = np.clip(sm_b[:, 1], 0, 1)
            boot_curves[b] = np.interp(x_grid, sm_b[:, 0], sm_b[:, 1],
                                        left=np.nan, right=np.nan)

        lower = np.nanpercentile(boot_curves, alpha * 100, axis=0)
        upper = np.nanpercentile(boot_curves, (1 - alpha) * 100, axis=0)

        br  = brier_score_loss(y_arr, p)
        ici = calibration_metrics(y_arr, p)["ici"]

        ax_main.plot(
            smooth_pts[:, 0], smooth_pts[:, 1],
            label=f"{m}  Brier={br:.4f}  ICI={ici:.4f}",
            **sty,
        )
        ax_main.fill_between(
            x_grid, lower, upper,
            alpha=0.13, color=sty["color"],
        )

    ax_main.set_ylabel("Observed event rate", fontsize=12)
    ax_main.set_ylim(ylim)
    ax_main.set_xlim(xlim)
    ax_main.legend(loc="upper left", fontsize=8.5, framealpha=0.92)
    ax_main.set_title(title, fontsize=12, pad=10)
    ax_main.grid(True, alpha=0.15)
    ax_main.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax_main.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

    # ── Histogram of predicted probabilities ──────────────────────────────────
    ax_hist.set_ylabel("N", fontsize=9)
    ax_hist.set_xlabel("Mean predicted probability", fontsize=12)
    ax_hist.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{int(x):,}"
    ))
    ax_hist.grid(True, alpha=0.10)

    # Plot stacked histograms (just first model for clarity, or all semi-transparent)
    colors_used = [_STYLE_MAP.get(m, {}).get("color", "steelblue")
                   for m in pred_dict]
    for (m, p_raw), clr in zip(pred_dict.items(), colors_used):
        p = np.clip(np.asarray(p_raw, dtype=float), 1e-7, 1 - 1e-7)
        ax_hist.hist(p, bins=np.linspace(xlim[0], xlim[1], 40),
                     alpha=0.25, color=clr, density=False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[SAVE] {save_path}")
    plt.show()
    plt.close()


print("[OK] Helper functions defined")

## Step 1. Pre-recalibration: smoothed calibration plots (original predictions)

In [ ]:
PLOT_DIR = RECAL_DIR / "plots"
PLOT_DIR.mkdir(exist_ok=True)

print("=== Pre-recalibration smoothed calibration plots ===")

for scen, pred_te, y_te, pred_ex, y_ex in [
    ("Scenario_A", pred_test_A, y_test_A, pred_ext_A, y_ext_A),
    ("Scenario_B", pred_test_B, y_test,   pred_ext_B, y_ext),
]:
    s = scen.replace("_", " ")

    smooth_calib_plot(
        pred_te, y_te,
        title=f"Smoothed Calibration — {s} | Internal test (before recalibration)",
        save_path=PLOT_DIR / f"smooth_calib_{scen.lower()}_internal_pre.png",
    )

    smooth_calib_plot(
        pred_ex, y_ex,
        title=f"Smoothed Calibration — {s} | External validation (before recalibration)",
        save_path=PLOT_DIR / f"smooth_calib_{scen.lower()}_external_pre.png",
    )

print("[DONE] Pre-recalibration plots saved")

In [ ]:
%whos

## Step 2. Calibration metrics table (before recalibration)

In [ ]:
print("=" * 72)
print("CALIBRATION METRICS — BEFORE RECALIBRATION")
print("=" * 72)
print()
print("Interpretation guide:")
print("  intercept ≈ 0  → mean predicted probability matches observed prevalence")
print("  intercept < 0  → model over-predicts (predicted > observed)")
print("  slope ≈ 1      → rank-order discrimination is well calibrated")
print("  ICI            → mean absolute calibration error (lower is better)")
print()

metrics_pre = {}
for scen, pred_te, y_te, pred_ex, y_ex in [
    ("Scenario_A", pred_test_A, y_test_A, pred_ext_A, y_ext_A),
    ("Scenario_B", pred_test_B, y_test,   pred_ext_B, y_ext),
]:
    df_int = calibration_metrics_table(
        pred_te, y_te,
        label=f"{scen} | Internal test"
    )
    df_ext = calibration_metrics_table(
        pred_ex, y_ex,
        label=f"{scen} | External validation"
    )

    metrics_pre[(scen, "internal")] = df_int
    metrics_pre[(scen, "external")] = df_ext

    df_int.to_csv(RECAL_DIR / f"{scen.lower()}_metrics_internal_pre.csv", index=False)
    df_ext.to_csv(RECAL_DIR / f"{scen.lower()}_metrics_external_pre.csv", index=False)

print("\n[SAVE] metrics saved to", RECAL_DIR)

## Step 3. Fit & apply intercept recalibration

**Fitting**: intercept is estimated on **internal test** predictions + outcomes.  
**Applying**: the same intercept is applied to shift external validation predictions.

In [ ]:
import json

print("=" * 72)
print("INTERCEPT RECALIBRATION")
print("=" * 72)

# ── Scenario A ────────────────────────────────────────────────────────────────
print("\n[Scenario A] Fitting on internal test → applying to external validation")
pred_ext_A_recal, intercepts_A = recalibrate_pred_dict(
    pred_dict_train  = pred_test_A,
    y_train          = y_test_A,
    pred_dict_target = pred_ext_A,
)

# Also recalibrate internal test itself for comparison
pred_test_A_recal, _ = recalibrate_pred_dict(
    pred_dict_train  = pred_test_A,
    y_train          = y_test_A,
    pred_dict_target = pred_test_A,
)

# ── Scenario B ────────────────────────────────────────────────────────────────
print("\n[Scenario B] Fitting on internal test → applying to external validation")
pred_ext_B_recal, intercepts_B = recalibrate_pred_dict(
    pred_dict_train  = pred_test_B,
    y_train          = y_test,
    pred_dict_target = pred_ext_B,
)

pred_test_B_recal, _ = recalibrate_pred_dict(
    pred_dict_train  = pred_test_B,
    y_train          = y_test,
    pred_dict_target = pred_test_B,
)

# ── Save intercepts ───────────────────────────────────────────────────────────
intercept_record = {
    "Scenario_A": intercepts_A,
    "Scenario_B": intercepts_B,
    "note": "Intercept fitted on internal test set predictions+outcomes, applied to external.",
}
with open(RECAL_DIR / "recalibration_intercepts.json", "w") as f:
    json.dump(intercept_record, f, indent=2)
print("\n[SAVE] recalibration_intercepts.json")

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for scen, ic in [("Scenario_A", intercepts_A), ("Scenario_B", intercepts_B)]:
    for m, val in ic.items():
        rows.append({"Scenario": scen, "Model": m, "Intercept": val})

df_intercepts = pd.DataFrame(rows)
print("\nFitted intercepts:")
print(df_intercepts.to_string(index=False))

## Step 4. Calibration metrics after recalibration

In [ ]:
print("=" * 72)
print("CALIBRATION METRICS — AFTER RECALIBRATION")
print("=" * 72)

metrics_post = {}
for scen, pred_te_r, y_te, pred_ex_r, y_ex in [
    ("Scenario_A", pred_test_A_recal, y_test_A, pred_ext_A_recal, y_ext_A),
    ("Scenario_B", pred_test_B_recal, y_test,   pred_ext_B_recal, y_ext),
]:
    df_int = calibration_metrics_table(
        pred_te_r, y_te,
        label=f"{scen} | Internal test (recalibrated)"
    )
    df_ext = calibration_metrics_table(
        pred_ex_r, y_ex,
        label=f"{scen} | External validation (recalibrated)"
    )
    metrics_post[(scen, "internal")] = df_int
    metrics_post[(scen, "external")] = df_ext

    df_int.to_csv(RECAL_DIR / f"{scen.lower()}_metrics_internal_post.csv", index=False)
    df_ext.to_csv(RECAL_DIR / f"{scen.lower()}_metrics_external_post.csv", index=False)

print("\n[SAVE] post-recalibration metrics saved")

## Step 5. Smoothed calibration plots after recalibration

In [ ]:
print("=== Post-recalibration smoothed calibration plots ===")

for scen, pred_te_r, y_te, pred_ex_r, y_ex in [
    ("Scenario_A", pred_test_A_recal, y_test_A, pred_ext_A_recal, y_ext_A),
    ("Scenario_B", pred_test_B_recal, y_test,   pred_ext_B_recal, y_ext),
]:
    s = scen.replace("_", " ")

    smooth_calib_plot(
        pred_te_r, y_te,
        title=f"Smoothed Calibration — {s} | Internal test (after recalibration)",
        save_path=PLOT_DIR / f"smooth_calib_{scen.lower()}_internal_post.png",
    )

    smooth_calib_plot(
        pred_ex_r, y_ex,
        title=f"Smoothed Calibration — {s} | External validation (after recalibration)",
        save_path=PLOT_DIR / f"smooth_calib_{scen.lower()}_external_post.png",
    )

print("[DONE] Post-recalibration plots saved")

## Step 6. Side-by-side comparison: before vs after recalibration (external)

Four-panel figure: [Scenario A external | Scenario B external] × [before | after]

In [ ]:
def smooth_calib_ax(
    ax, pred_dict, y_true, title,
    frac=0.30, n_bootstrap=500, ci=0.95,
    xlim=(0.0, 0.25), ylim=(-0.02, 0.40),
):
    """Draw smoothed calibration on a provided Axes object (for subplot use)."""
    y_arr  = np.asarray(y_true, dtype=float)
    alpha  = (1 - ci) / 2
    x_grid = np.linspace(xlim[0] + 0.001, xlim[1], 300)
    rng    = np.random.default_rng(42)

    x_ref = np.linspace(*xlim, 300)
    ax.plot(x_ref, x_ref, "k--", lw=1.0, alpha=0.50)

    for m, p_raw in pred_dict.items():
        p   = np.clip(np.asarray(p_raw, dtype=float), 1e-7, 1 - 1e-7)
        sty = _STYLE_MAP.get(m, dict(ls="-", lw=2, color="steelblue"))
        mask = (p >= xlim[0]) & (p <= xlim[1])
        p_m, y_m = p[mask], y_arr[mask]
        if len(p_m) < 10:
            continue

        sort_idx   = np.argsort(p_m)
        smooth_pts = lowess(y_m[sort_idx], p_m[sort_idx],
                            frac=frac, it=3, return_sorted=True)
        smooth_pts[:, 1] = np.clip(smooth_pts[:, 1], 0, 1)

        boot_curves = np.empty((n_bootstrap, len(x_grid)))
        for b in range(n_bootstrap):
            idx_b    = rng.integers(0, len(p_m), len(p_m))
            p_b, y_b = p_m[idx_b], y_m[idx_b]
            sort_b   = np.argsort(p_b)
            sm_b     = lowess(y_b[sort_b], p_b[sort_b],
                              frac=frac, it=1, return_sorted=True)
            sm_b[:, 1] = np.clip(sm_b[:, 1], 0, 1)
            boot_curves[b] = np.interp(x_grid, sm_b[:, 0], sm_b[:, 1],
                                        left=np.nan, right=np.nan)

        lower = np.nanpercentile(boot_curves, alpha * 100, axis=0)
        upper = np.nanpercentile(boot_curves, (1 - alpha) * 100, axis=0)

        br  = brier_score_loss(y_arr, p)
        ax.plot(smooth_pts[:, 0], smooth_pts[:, 1],
                label=f"{m} (Brier={br:.4f})", **sty)
        ax.fill_between(x_grid, lower, upper,
                         alpha=0.12, color=sty["color"])

    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.12)
    ax.legend(loc="upper left", fontsize=7.5, framealpha=0.9)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))


fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for row, (scen, pred_ex_pre, pred_ex_post, y_ex) in enumerate([
    ("Scenario A", pred_ext_A,      pred_ext_A_recal, y_ext_A),
    ("Scenario B", pred_ext_B,      pred_ext_B_recal, y_ext),
]):
    smooth_calib_ax(
        axes[row, 0], pred_ex_pre,  y_ex,
        title=f"{scen} | External — Before recalibration",
    )
    smooth_calib_ax(
        axes[row, 1], pred_ex_post, y_ex,
        title=f"{scen} | External — After recalibration",
    )

for ax in axes[:, 0]: ax.set_ylabel("Observed event rate", fontsize=11)
for ax in axes[1, :]: ax.set_xlabel("Mean predicted probability", fontsize=11)

plt.suptitle("External Validation: Smoothed Calibration Before vs After Intercept Recalibration",
             fontsize=12, y=1.01)
plt.tight_layout()
comp_path = PLOT_DIR / "comparison_pre_post_recalibration_external.png"
plt.savefig(comp_path, dpi=150, bbox_inches="tight")
print(f"[SAVE] {comp_path}")
plt.show()

## Step 7. Combined summary table (before vs after, all scenarios)

In [ ]:
rows = []
for scen, pred_ex_pre, y_ex, pred_ex_post in [
    ("Scenario_A", pred_ext_A, y_ext_A, pred_ext_A_recal),
    ("Scenario_B", pred_ext_B, y_ext,   pred_ext_B_recal),
]:
    for m in pred_ex_pre:
        m_pre  = calibration_metrics(y_ex, pred_ex_pre[m])
        m_post = calibration_metrics(y_ex, pred_ex_post[m])
        rows.append({
            "Scenario"    : scen,
            "Model"       : m,
            "intercept_pre"  : m_pre["intercept"],
            "intercept_post" : m_post["intercept"],
            "slope_pre"      : m_pre["slope"],
            "slope_post"     : m_post["slope"],
            "ici_pre"        : m_pre["ici"],
            "ici_post"       : m_post["ici"],
            "brier_pre"      : m_pre["brier"],
            "brier_post"     : m_post["brier"],
        })

df_summary = pd.DataFrame(rows)

print("\n" + "=" * 80)
print("EXTERNAL VALIDATION CALIBRATION SUMMARY: Before vs After Recalibration")
print("=" * 80)
print(df_summary.to_string(index=False))

out_path = RECAL_DIR / "calibration_summary_external_pre_post.csv"
df_summary.to_csv(out_path, index=False)
print(f"\n[SAVE] {out_path}")

## Step 8. Save recalibrated predictions to disk

In [ ]:
import joblib

for tag, obj in [
    ("pred_ext_A_recal",   pred_ext_A_recal),
    ("pred_ext_B_recal",   pred_ext_B_recal),
    ("pred_test_A_recal",  pred_test_A_recal),
    ("pred_test_B_recal",  pred_test_B_recal),
]:
    path = RECAL_DIR / f"{tag}.joblib"
    joblib.dump(obj, path, compress=3)
    print(f"[SAVE] {path.name}  ({path.stat().st_size/1024:.0f} KB)")

print("\n[DONE] All recalibration outputs saved to:", RECAL_DIR)